In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
len(words) #dataset size

32033

In [5]:
chars = sorted(list(set(''.join(words)))) #build the vocabulary of characters
stoi = {s:i+1 for i,s in enumerate(chars)} #letter to integer(index) mapping    
stoi['.'] = 0 #index 0 will be reserved for the end of a word and start of the word
itos = {i:s for s,i in stoi.items()} # reverse mapping from integer(index) to letter
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [6]:
block_size = 3 # context length: how many characters do we take to predict the next one? #upgrade to bigram level
X, Y = [], [] #inputs and targets split variables
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size 
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [7]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [8]:
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [9]:
C = torch.randn((27, 2))

In [10]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [11]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [12]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [13]:
h

tensor([[ 0.4065, -0.6587,  0.4340,  ...,  0.9985, -0.9825,  0.9987],
        [ 0.9953, -0.9907,  0.9999,  ...,  0.3147,  0.5292,  0.1515],
        [ 0.9801, -0.9332,  0.9826,  ..., -0.9916,  0.9417,  0.9954],
        ...,
        [-0.8434, -0.8874, -0.8001,  ...,  0.9219,  0.7108, -0.9634],
        [ 0.9805, -0.9993,  0.9922,  ...,  0.2221,  0.5998,  0.9202],
        [-0.9394,  0.9989,  0.9530,  ..., -0.9866,  0.9612,  0.9872]])

In [14]:
h.shape

torch.Size([32, 100])